# 03 Validation

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir("/content/drive/MyDrive/Gates Foundation/Building Dataset Validation/")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/Gates Foundation/Building Dataset Validation")
CONFIG_PATH  = PROJECT_ROOT / "configs/validation_configs.yaml"

# Set overwrite=True to re-run cities whose outputs already exist.
# When False (default), cities with existing sentinel parquets are skipped automatically.
OVERWRITE = True

In [3]:
import sys, os
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [4]:
import logging
import yaml
from src.validator import UrbanValidator

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

# Patch overwrite flag into config before constructing Validator
with open(CONFIG_PATH) as f:
    _cfg_preview = yaml.safe_load(f)

datasets_preview = _cfg_preview.get("vector", {}).get("datasets", [])
enabled = [d["name"] for d in datasets_preview if d.get("enabled", True)]
print(f"Config: {CONFIG_PATH}")
print(f"Enabled candidate datasets: {enabled}")

Config: /content/drive/MyDrive/Gates Foundation/Building Dataset Validation/configs/validation_configs.yaml
Enabled candidate datasets: ['overture', 'gba', 'globfp']


In [5]:
# Instantiate the validator — this reads the config and AOI tracker,
# resolves file paths, and logs how many datasets are queued.
# The overwrite flag from the cell above is injected before loading.
import yaml, copy

with open(CONFIG_PATH) as f:
    _cfg_patched = yaml.safe_load(f)
_cfg_patched.setdefault("output", {})["overwrite"] = OVERWRITE

# Write patched config to a temp file so Validator can read it
import tempfile, os
_tmp = tempfile.NamedTemporaryFile(
    mode="w", suffix=".yaml", delete=False, dir=PROJECT_ROOT / "configs"
)
yaml.dump(_cfg_patched, _tmp)
_tmp.close()
_PATCHED_CONFIG_PATH = _tmp.name

v = UrbanValidator(_PATCHED_CONFIG_PATH)
print(f"\nDatasets queued: {len(v.datasets)}")
for ds in v.datasets:
    print(f"  {ds['id']}")

00:20:07  INFO      Validation tracker: 70 -> 70 suitable rows.
00:20:10  INFO      Loaded 70 dataset(s) for validation.
00:20:10  INFO      Loaded 70 dataset(s) for validation.



Datasets queued: 70
  ant-curacao
  bgd-rohingya
  blz-burrell-boom
  bra-nova-sussuarana
  col-san-antonio-de-prado
  cvg-saint-vincent-grenadines
  dom-dominica
  gha-accra
  gha-aiyim-sraha
  gha-dansoman
  gha-nawuni
  gha-sawla-tuna
  gha-wa
  grd-grenada
  jam-kingston
  jam-saint-catherine
  jpn-ashiya-hama
  jpn-hiroshima
  jpn-iwate
  jpn-izu-oshima
  jpn-kashima
  jpn-numakunai
  ken-kakuma
  ken-kakuma-kalobeyei
  ken-mukuru
  ken-nairobi
  lbr-monrovia
  lby-almarj
  lby-bayda
  lby-darnah
  lby-susah
  lca-saint-lucia
  maf-saint-martin
  mmr-patheingyi-mandalay
  moz-de-maio
  moz-djonasse
  mwi-lilongwe
  mwi-mlowe
  ner-niame
  nga-ibadan
  phl-bagamanoc
  phl-barangay
  phl-catanduanes
  phl-juraojurao-anini-y-antique
  phl-pasig
  phl-poblacion-sagua-anini-y-antique
  phl-viga
  phl-visayas
  sle-cockle-bay-1
  sle-cockle-bay-2
  sle-freetown
  sle-kolleh
  sle-kroo-bay-1
  ssd-juba
  swz-nhlangano
  sxm-sint-maarten
  tjk-artuch
  tjk-nomandiyon
  tjk-tavishi-bolo
 

In [ ]:
import pandas as pd

results = v.validate_vector()

# Clean up temp config
try:
    os.unlink(_PATCHED_CONFIG_PATH)
except Exception:
    pass

# Summary
summary = pd.DataFrame(
    [{"aoi": k, "status": "ok" if v else "failed"} for k, v in results.items()]
)
print(f"\nDone — {len(summary)} AOIs processed.\n")
display(summary.groupby("status")["aoi"].count().rename("count").to_frame())
display(summary)

00:20:10  INFO      ━━━━  ant-curacao  ━━━━
00:20:10  INFO      MEM [ant-curacao start] RSS = 342 MB
00:20:10  INFO      [ant-curacao] Working CRS: EPSG:32619 | 57 tiles.
00:20:10  INFO      Created 57 records
00:20:11  INFO      [ant-curacao] Reference buildings: 15054 (from 1 file(s))
00:20:11  INFO      [ant-curacao / overture] Candidate: ant_curacao_overture_building.parquet
00:20:12  INFO      [ant-curacao / overture] Candidate buildings: 19169
00:20:42  INFO      [ant-curacao / overture] Tile metrics saved → vector_metrics_tiles_overture.parquet
00:20:42  INFO      MEM [ant-curacao/overture done] RSS = 500 MB
00:20:42  INFO      [ant-curacao / gba] Candidate: ant_curacao_gba.parquet
00:20:42  ERROR     [ant-curacao] Unhandled error during vector validation.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/geopandas/io/arrow.py", line 654, in _read_parquet_schema_and_metadata
    schema = parquet.ParquetDataset(path, filesystem=filesystem, **kwarg

[grenada_ref.geojson] fixing 1 invalid geometries...


00:49:01  INFO      [grd-grenada] Reference buildings: 50074 (from 1 file(s))
00:49:02  WARNING   [grd-grenada / overture] No candidate files found (pattern: grd_grenada_overture*.parquet).
00:49:02  WARNING   [grd-grenada / gba] No candidate files found (pattern: grd_grenada_gba*.parquet).
00:49:02  WARNING   [grd-grenada / globfp] No candidate files found (pattern: grd_grenada_globfp*.parquet).
00:49:02  WARNING   [grd-grenada] No tile metrics produced — skipping output.
00:49:02  INFO      ━━━━  jam-kingston  ━━━━
00:49:02  INFO      MEM [jam-kingston start] RSS = 1177 MB
00:49:02  INFO      [jam-kingston] Working CRS: EPSG:32618 | 345 tiles.
00:49:02  INFO      Created 345 records
00:49:08  INFO      [jam-kingston] Reference buildings: 122846 (from 1 file(s))
00:49:08  INFO      [jam-kingston / overture] Candidate: jam_kingston_overture_building.parquet
00:49:11  INFO      [jam-kingston / overture] Candidate buildings: 161026
00:53:53  INFO      [jam-kingston / overture] Tile metri